# Exercise 3: Neural networks in PyTorch

In this exercise you’ll implement small neural-network building blocks from scratch and use them to train a simple classifier.

You’ll cover:
- **Basic layers**: Linear, Embedding, Dropout
- **Normalization**: LayerNorm and RMSNorm
- **MLPs + residual**: composing layers into deeper networks
- **Classification**: generating a learnable dataset, implementing cross-entropy from logits, and writing a minimal training loop

As before: fill in all `TODO`s without changing function names or signatures.
Use small sanity checks and compare to PyTorch reference implementations when useful.

In [39]:
from __future__ import annotations

import torch
from torch import nn

## Basic layers

In this section you’ll implement a few core layers that appear everywhere:

### `Linear`
A fully-connected layer that follows nn.Linear conventions:  
`y = x @ Wᵀ + b`

Important details:
- Parameters should be registered as `nn.Parameter`
- Store weight as (out_features, in_features) like nn.Linear.
- The forward pass should support leading batch dimensions: `x` can be shape `(..., in_features)`

### `Embedding`
An embedding table maps integer ids to vectors:
- input: token ids `idx` of shape `(...,)`
- output: vectors of shape `(..., embedding_dim)`

This is essentially a learnable lookup table.

### `Dropout`
Dropout randomly zeroes activations during training to reduce overfitting.
Implementation details:
- Only active in `model.train()` mode
- In training: drop with probability `p` and scale the kept values by `1/(1-p)` so the expected value stays the same
- In eval: return the input unchanged

## Instructions
- Do not use PyTorch reference modules for the parts you implement (e.g. don’t call nn.Linear inside your Linear).
- You may use standard tensor ops that you learned before (matmul, sum, mean, rsqrt, indexing, etc.).
- Use a parameter initialization method of your choice. We recommend something like Xavier-uniform.


In [40]:
class Linear(nn.Module):
    def __init__(self, in_features: int, out_features: int, bias: bool = True):
        super().__init__()
        # TODO: initialize parameters
        bound =  (6 / (in_features + out_features)) ** 0.5 # we use xavier uniform
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        
        with torch.no_grad():
            self.weight.uniform_(-bound, bound)
        if bias:
            self.bias = nn.Parameter(torch.empty(out_features))
            with torch.no_grad():
                self.bias.zero_()
        else:
            self.bias = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (..., in_features)
        return: (..., out_features)
        """
        # TODO: implement
        out = x @ self.weight.T
        if self.bias is not None:
            out += self.bias
        return out
    
x = torch.randn(4, 5)
linear = Linear(5, 3)
out_torch = nn.functional.linear(x, linear.weight, linear.bias)
out = linear.forward(x)
print(out_torch == out)

tensor([[True, True, True],
        [True, True, True],
        [True, True, True],
        [True, True, True]])


In [41]:
class Embedding(nn.Module):
    def __init__(self, num_embeddings: int, embedding_dim: int):
        super().__init__()
        # TODO: initialize
        bound = (6 / (num_embeddings + embedding_dim)) ** 0.5 # we use xavier uniform
        self.weight = nn.Parameter(torch.empty(num_embeddings, embedding_dim))
        with torch.no_grad():
            self.weight.uniform_(-bound, bound)

        

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """
        idx: (...,) int64
        return: (..., embedding_dim)
        """
        # TODO: implement (index into weight)
        return self.weight[idx]

In [42]:
class Dropout(nn.Module):
    def __init__(self, p: float):
        super().__init__()
        self.p = p

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        In train mode: drop with prob p and scale by 1/(1-p).
        In eval mode: return x unchanged.
        """
        # TODO: implement without using nn.Dropout
        if self.training:
            mask = torch.rand(x.shape) > self.p
            return x * mask / (1 - self.p)
        else:
            return x
            


x = torch.randn(4, 5)
dropout = Dropout(0.5)
dropout.train()
out = dropout.forward(x)
out

tensor([[-0.0000, -0.0000, -0.0000, -0.0000, 0.0000],
        [0.0000, 0.9338, 3.9013, -0.0000, 2.2807],
        [-0.0000, 0.0000, -0.0000, -0.0000, -0.0000],
        [-0.0000, 0.0000, 0.0000, 2.2025, -0.0000]])

## Normalization

Normalization layers help stabilize training by controlling activation statistics.

### LayerNorm
LayerNorm normalizes each example across its **feature dimension** (the last dimension):

- compute mean and variance over the last dimension
- normalize: `(x - mean) / sqrt(var + eps)`
- apply learnable per-feature scale and shift (`weight`, `bias`)

**In this exercise, assume `elementwise_affine=True` (always include `weight` and `bias`).**  
`weight` and `bias` each have shape `(D,)`.

LayerNorm is widely used in transformers because it does not depend on batch statistics.

### RMSNorm
RMSNorm is similar to LayerNorm but normalizes using only the root-mean-square:
- `x / sqrt(mean(x^2) + eps)` over the last dimension
- usually includes a learnable scale (`weight`)
- no mean subtraction

RMSNorm is popular in modern LLMs because it's faster.


In [43]:
class LayerNorm(nn.Module):
    def __init__(
        self, normalized_shape: int, eps: float = 1e-5, elementwise_affine: bool = True
    ):
        super().__init__()
        # TODO: implement
        self.eps = eps
        self.elementwise_affine = elementwise_affine
        if elementwise_affine:
            self.weight = nn.Parameter(torch.ones(normalized_shape))
            self.bias = nn.Parameter(torch.zeros(normalized_shape))
        else:
            self.register_parameter('weight', None) # in case elementwise_affine is false but should not be in this exercise
            self.register_parameter('bias', None)
        

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Normalize over the last dimension.
        x: (..., D)
        """
        # TODO: implement
        var, mean = torch.var_mean(x, dim=-1, keepdim=True, unbiased=False) # only over last dim
        ret = (x - mean) / ((var + self.eps) ** 0.5)
        if self.elementwise_affine:
            ret = ret * self.weight + self.bias
        return ret
        

        
# x = torch.tensor([[1.0,1.0,2.0,2.0,2.0], [2.0,1.0,2.0,2.0,2.0]])
x = torch.randn(1, 1, 2, 2)
norm_layer = LayerNorm(x.shape[-1])
out = norm_layer.forward(x)
out_torch = nn.LayerNorm(x.shape[-1], eps=norm_layer.eps, elementwise_affine=norm_layer.elementwise_affine).forward(x)
print(out_torch)
print(out)

tensor([[[[-1.0000,  1.0000],
          [-1.0000,  1.0000]]]], grad_fn=<NativeLayerNormBackward0>)
tensor([[[[-1.0000,  1.0000],
          [-1.0000,  1.0000]]]], grad_fn=<AddBackward0>)


In [44]:
class RMSNorm(nn.Module):
    def __init__(self, normalized_shape: int, eps: float = 1e-8):
        super().__init__()
        # TODO: implement
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(normalized_shape))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        RMSNorm: x / sqrt(mean(x^2) + eps) * weight
        over the last dimension.
        """
        # TODO: implement
        return x / ((torch.mean(x*x) + self.eps) ** 0.5) * self.weight
    
x = torch.tensor([[2.0,1.0,2.0,2.0,2.0], [2.0,1.0,2.0,2.0,2.0]])
norm_layer = RMSNorm(x.shape[-1])
out_torch = nn.RMSNorm(x.shape[-1], eps=norm_layer.eps).forward(x)
out = norm_layer.forward(x)
print(out_torch)
print(out)


tensor([[1.0847, 0.5423, 1.0847, 1.0847, 1.0847],
        [1.0847, 0.5423, 1.0847, 1.0847, 1.0847]], grad_fn=<MulBackward0>)
tensor([[1.0847, 0.5423, 1.0847, 1.0847, 1.0847],
        [1.0847, 0.5423, 1.0847, 1.0847, 1.0847]], grad_fn=<MulBackward0>)


## MLPs and residual networks

Now you’ll build larger networks by composing layers.

### MLP
An MLP is a stack of `depth` Linear layers with non-linear activations (use GELU) in between.
In this exercise you’ll support:
- configurable depth
- a hidden dimension
- optional LayerNorm between layers (a common stabilization trick)

A key skill is building networks using `nn.ModuleList` / `nn.Sequential` while keeping shapes consistent.

### Transformer-style FeedForward (FFN)
A transformer block contains a position-wise feedforward network:
- `D -> 4D -> D` (by default)
- activation is typically **GELU**

This is essentially an MLP applied independently at each token position.

### Residual wrapper
Residual connections are the simplest form of “skip connection”:
- output is `x + fn(x)`

They improve gradient flow and allow training deeper networks more reliably.

In [45]:
class MLP(nn.Module):
    def __init__(
        self,
        in_dim: int,
        hidden_dim: int,
        out_dim: int,
        depth: int,
        use_layernorm: bool = False,
    ):
        super().__init__()
        # TODO: build modules (list of Linear + activation)
        # Optionally insert LayerNorm between layers.
        if depth < 2:
            raise ValueError("Cannot constuct proper MLP with depth < 2")
        layers_ = [Linear(in_dim, hidden_dim)] # we use our own
        if use_layernorm:
            layers_.append(LayerNorm(hidden_dim))
        for i in range(depth - 2):
            layers_.append(nn.GELU())
            layers_.append(Linear(hidden_dim, hidden_dim))
            if use_layernorm:
                layers_.append(LayerNorm(hidden_dim))
        layers_.append(nn.GELU())    
        layers_.append(Linear(hidden_dim, out_dim))
        self.layers = nn.Sequential(*layers_)

            
        

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: implement
        return self.layers(x)
    
mlp = MLP(10, 40, 10, 4, use_layernorm=True)
print(mlp.layers)

x = torch.randn([3,4,5,10])
out = mlp.forward(x)
print(out)
print(out.shape)

Sequential(
  (0): Linear()
  (1): LayerNorm()
  (2): GELU(approximate='none')
  (3): Linear()
  (4): LayerNorm()
  (5): GELU(approximate='none')
  (6): Linear()
  (7): LayerNorm()
  (8): GELU(approximate='none')
  (9): Linear()
)
tensor([[[[ 1.0714, -0.1801,  0.7625, -0.2746,  0.2485,  0.5128, -0.2537,
            0.5517,  0.5817,  0.4290],
          [ 0.7183, -0.5042,  1.0374,  0.5730,  0.5531,  1.0179, -0.2633,
           -1.0012, -0.4252, -1.0778],
          [ 0.9075, -0.3137,  0.5938, -0.1762,  0.9347,  1.0420, -0.4484,
           -0.2746,  0.5876,  0.3437],
          [ 0.8934, -0.7842,  0.7587, -0.6561,  0.6803,  1.2105,  0.2314,
            0.9745,  0.8780,  0.6563],
          [ 1.0173, -0.1967,  0.7613, -0.0739,  1.1775,  0.4792, -0.1915,
            0.6067,  0.7621,  0.6355]],

         [[ 0.9219,  0.0658,  0.3938, -0.6099,  0.7710,  0.3498,  0.1869,
            0.1154,  0.9763,  0.2209],
          [ 0.8942, -1.2106,  1.1063,  0.8384,  0.3776,  0.3443, -0.0979,
           -0.1

In [46]:
class FeedForward(nn.Module):
    """
    Transformer-style FFN: D -> 4D -> D (default)
    """

    def __init__(self, d_model: int, d_ff: int | None = None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        # TODO: create two Linear layers and choose an activation (GELU)
        self.linear1 = Linear(d_model, d_ff)
        self.linear2 = Linear(d_ff, d_model)
        self.activation = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: implement
        ret = self.linear1(x)
        ret = self.activation(ret)
        ret = self.linear2(ret)
        return ret

x = torch.randn([3,4,5,10])
feedforward = FeedForward(10)
out = feedforward.forward(x)
print(out)
print(out.shape)

tensor([[[[ 1.6934e-01,  1.4965e-01, -3.7625e-02,  1.2024e-01, -4.1942e-01,
           -3.5562e-02,  6.2072e-01,  4.3865e-01, -5.0007e-01,  2.7555e-02],
          [ 7.7172e-02,  1.4177e-01, -1.4161e+00, -1.9492e-01,  5.4168e-01,
            2.0676e-01, -4.5719e-03,  8.7616e-01, -7.4216e-01,  7.5500e-01],
          [ 5.9979e-01, -4.8215e-02, -9.6773e-01,  3.5355e-01,  9.2442e-01,
            4.0124e-01, -8.0753e-02,  5.0329e-01,  5.0850e-02, -9.9667e-02],
          [ 9.3422e-01,  2.1074e-01, -4.3426e-01,  6.4824e-01,  3.1151e-01,
            4.0754e-01,  6.0918e-01,  1.2351e-01, -7.4060e-01, -1.3443e-01],
          [ 2.3616e-02,  3.8022e-01, -8.0037e-01, -2.8071e-01,  7.1496e-01,
           -2.9632e-01, -1.3503e-01,  2.8086e-01, -5.0179e-01,  2.7456e-01]],

         [[ 3.9670e-01,  1.0195e+00, -1.4812e+00,  5.9580e-01,  3.2387e-01,
            6.8637e-01,  1.0149e+00,  4.6228e-01, -5.8530e-01,  9.2772e-02],
          [ 4.6818e-01,  4.2051e-01, -2.8384e-03,  7.7500e-02,  3.2170e-01,
    

In [47]:
class Residual(nn.Module):
    def __init__(self, fn: nn.Module):
        super().__init__()
        # TODO: implement
        self.fn = fn

    def forward(self, x: torch.Tensor, *args, **kwargs) -> torch.Tensor:
        # TODO: return x + fn(x, ...)
        return x + self.fn(x, *args, **kwargs)


## Classification problem

In this section you’ll put everything together in a minimal MNIST classification experiment.

You will:
1) download and load the MNIST dataset
2) implement cross-entropy from logits (stable, using log-softmax)
3) build a simple MLP-based classifier (flatten MNIST images first)
4) write a minimal training loop
5) report train loss curve and final accuracy

The goal here is not to reach state-of-the-art accuracy, but to understand the full pipeline:
data → model → logits → loss → gradients → parameter update.

### Model notes
- We want you to combine the MLP we implemented above with the classification head we define below into one model 

### MNIST notes
- MNIST images are `28×28` grayscale.
- After `ToTensor()`, each image has shape `(1, 28, 28)` and values in `[0, 1]`.
- For an MLP classifier, we flatten to a vector of length `784`.

## Deliverables
- Include a plot of your train loss curve in the video submission as well as a final accuracy. 
- **NOTE** Here we don't grade on model performance but we expect you to achieve at least 70% accuracy to confirm a correct model implementation.

In [48]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [49]:
transform = transforms.ToTensor()  # -> float32 in [0,1], shape (1, 28, 28)

train_ds = datasets.MNIST(root="data", train=True, download=True, transform=transform)
test_ds  = datasets.MNIST(root="data", train=False, download=True, transform=transform)

# TODO: define the dataloaders
dataloader_train = DataLoader(train_ds, batch_size=8, shuffle=True)
dataloader_test = DataLoader(test_ds, batch_size=8, shuffle=False)


print(next(iter(dataloader_test))[0].shape)

torch.Size([8, 1, 28, 28])


In [50]:
def cross_entropy_from_logits(
    logits: torch.Tensor,
    targets: torch.Tensor,
) -> torch.Tensor:
    """
    Compute mean cross-entropy loss from logits.

    logits: (B, C)
    targets: (B,) int64

    Requirements:
    - Use log-softmax for stability (do not use torch.nn.CrossEntropyLoss, we check this in the autograder).
    """
    # TODO: implement
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
    loss = torch.nn.functional.nll_loss(log_probs, targets)
    return loss

x = torch.tensor([[4.0, -10.0, 0.0]])
y = torch.tensor([0]) 
cross_entropy_from_logits(x, y)

tensor(0.0182)

In [51]:
class ClassificationHead(nn.Module):
    def __init__(self, d_in: int, num_classes: int):
        super().__init__()
        # TODO: implement
        self.layer = Linear(d_in, num_classes)
        

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (..., d_in)
        return: (..., num_classes) logits
        """
        # TODO: implement
        return self.layer(x)

x = torch.tensor([[4.0, -10.0, 0.0]])
head = ClassificationHead(3,2)
head(x)

tensor([[7.2647, 2.6955]], grad_fn=<AddBackward0>)

In [52]:
def accuracy(loader, model): # could not see how to do it without adding model into params
    # TODO: You can use this function to evaluate your model accuracy.
    model.eval() # not sure but i guess model is assumend to be global
    correct = 0
    total = 0

    with torch.no_grad():
        for data in loader:
            inputs, labels = data
            y_hat = model(inputs)
            _, predicted_class = torch.max(y_hat.data, 1)
            total += labels.size(0)
            correct += (predicted_class == labels).sum().item()
    
    return correct / total


In [59]:
# %run ex2.ipynb # to use own optimizer

def train_classifier(
    model: nn.Module,
    train_data_loader: DataLoader,
    test_data_loader: DataLoader,
    lr: float,
    epochs: int,
    seed: int = 0,
) -> list[float]:
    """
    Minimal training loop for MNIST classification.

    Steps:
    - define optimizer
    - for each epoch:
        - sample minibatches
        - forward -> cross-entropy -> backward -> optimizer step
      - compute test accuracy at the end of each epoch
    - return list of training losses (one per update step)

    Requirements:
    - call model.train() during training and model.eval() during evaluation
    - do not use torch.nn.CrossEntropyLoss (use your cross_entropy_from_logits)
    """
    # TODO: implement

    torch.manual_seed(seed)

    # wanna try own optimizer
    
    # params = [param for param in model.parameters() if param.requires_grad]
    # state = [init_adamw_state(p) for p in params]

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    

    for epoch in range(epochs):
        model.train()
        losses = []
        for i, batch in enumerate(train_data_loader):
            x, y = batch
            y_hat = model(x)
            loss = cross_entropy_from_logits(y_hat, y)
            losses.append(loss.item())

            loss.backward()
            # if i % 1000 == 0:
                # print(f"loss:  {loss.item()}")
            # grads = [p.grad for p in params]
            # if any(g is None for g in grads):
            #     print("None grads detected")
            # adamw_step_many_(params, grads, state, lr=lr)

            # for p in params:
            #     p.grad = None

            optimizer.step()
            optimizer.zero_grad()

        acc = accuracy(test_data_loader, model)
        print(f"Epoch {epoch}, Accuracy: {acc}, Train Loss: {sum(losses)/len(losses)}")


In [54]:

class MNIST_classifier(nn.Module):
    def __init__(self, hidden_layers, depth, use_layernorm=True):
        super().__init__()
        self.layer = nn.Sequential(
            MLP(784, hidden_layers, 64, depth, use_layernorm=use_layernorm),
            ClassificationHead(64, 10)
        )

    def forward(self, x):
        x = x.flatten(start_dim=1)
        return self.layer(x)


In [55]:
train_classifier(
    model=MNIST_classifier(96, 3),
    train_data_loader=dataloader_train,
    test_data_loader=dataloader_test,
    lr=1e-4,
    epochs=20,
)

Epoch 0, Accuracy: 0.9581, Train Loss: 0.2908079554107971
Epoch 1, Accuracy: 0.969, Train Loss: 0.11141924398888368
Epoch 2, Accuracy: 0.9727, Train Loss: 0.07652099247696266
Epoch 3, Accuracy: 0.9723, Train Loss: 0.05742965122972188
Epoch 4, Accuracy: 0.9748, Train Loss: 0.04431416174542101
Epoch 5, Accuracy: 0.9727, Train Loss: 0.03533374192639603
Epoch 6, Accuracy: 0.976, Train Loss: 0.029005834294765494
Epoch 7, Accuracy: 0.9761, Train Loss: 0.02350844856432195
Epoch 8, Accuracy: 0.9743, Train Loss: 0.018928761369083804
Epoch 9, Accuracy: 0.9733, Train Loss: 0.016538545532038325
Epoch 10, Accuracy: 0.9762, Train Loss: 0.013439012654393945
Epoch 11, Accuracy: 0.9768, Train Loss: 0.012255922519566927
Epoch 12, Accuracy: 0.9767, Train Loss: 0.009866809688546634
Epoch 13, Accuracy: 0.9789, Train Loss: 0.008897384242045246
Epoch 14, Accuracy: 0.9731, Train Loss: 0.008061176954508467
Epoch 15, Accuracy: 0.9786, Train Loss: 0.007487310157817054
Epoch 16, Accuracy: 0.9788, Train Loss: 0.00

In [56]:
train_classifier(
    model=MNIST_classifier(96, 3, use_layernorm=False),
    train_data_loader=dataloader_train,
    test_data_loader=dataloader_test,
    lr=1e-4,
    epochs=20,
)

Epoch 0, Accuracy: 0.939, Train Loss: 0.36906169177067155
Epoch 1, Accuracy: 0.9565, Train Loss: 0.17259122805176302
Epoch 2, Accuracy: 0.9633, Train Loss: 0.12581287893740228
Epoch 3, Accuracy: 0.9689, Train Loss: 0.10017619108338065
Epoch 4, Accuracy: 0.9716, Train Loss: 0.08144965479029342
Epoch 5, Accuracy: 0.9747, Train Loss: 0.06967745025761785
Epoch 6, Accuracy: 0.9768, Train Loss: 0.05919589216805216
Epoch 7, Accuracy: 0.9761, Train Loss: 0.05093734725402404
Epoch 8, Accuracy: 0.9761, Train Loss: 0.04449598363756753
Epoch 9, Accuracy: 0.9757, Train Loss: 0.03891711116501783
Epoch 10, Accuracy: 0.9777, Train Loss: 0.033713595700390044
Epoch 11, Accuracy: 0.9785, Train Loss: 0.030145481252137247
Epoch 12, Accuracy: 0.9744, Train Loss: 0.026474633691897443
Epoch 13, Accuracy: 0.977, Train Loss: 0.022599470844862298
Epoch 14, Accuracy: 0.9792, Train Loss: 0.020014841937307536
Epoch 15, Accuracy: 0.9764, Train Loss: 0.017441109691818623
Epoch 16, Accuracy: 0.9788, Train Loss: 0.0159

In [57]:
train_classifier(
    model=MNIST_classifier(64, 2, use_layernorm=False),
    train_data_loader=dataloader_train,
    test_data_loader=dataloader_test,
    lr=1e-4,
    epochs=20,
)

Epoch 0, Accuracy: 0.922, Train Loss: 0.43694732115653656
Epoch 1, Accuracy: 0.9462, Train Loss: 0.22023666011647633
Epoch 2, Accuracy: 0.9577, Train Loss: 0.16755610719917652
Epoch 3, Accuracy: 0.9607, Train Loss: 0.13670733340108612
Epoch 4, Accuracy: 0.9653, Train Loss: 0.11617678038303275
Epoch 5, Accuracy: 0.9666, Train Loss: 0.10118312104888998
Epoch 6, Accuracy: 0.9698, Train Loss: 0.08948522388185132
Epoch 7, Accuracy: 0.97, Train Loss: 0.07968964235179786
Epoch 8, Accuracy: 0.9716, Train Loss: 0.07189799352055028
Epoch 9, Accuracy: 0.9728, Train Loss: 0.06458840938825354
Epoch 10, Accuracy: 0.9743, Train Loss: 0.05891040634860304
Epoch 11, Accuracy: 0.9746, Train Loss: 0.053999225796498164
Epoch 12, Accuracy: 0.9736, Train Loss: 0.04893018278717257
Epoch 13, Accuracy: 0.975, Train Loss: 0.0444786175676271
Epoch 14, Accuracy: 0.9763, Train Loss: 0.04167771737572354
Epoch 15, Accuracy: 0.9765, Train Loss: 0.03791514552898613
Epoch 16, Accuracy: 0.9747, Train Loss: 0.035082125243

In [ ]:
train_classifier(
    model=MNIST_classifier(64, 2, use_layernorm=False),
    train_data_loader=dataloader_train,
    test_data_loader=dataloader_test,
    lr=1e-4,
    epochs=20,
) # had to remove use of own optimizer from ex2

Epoch 0, Accuracy: 0.9259, Train Loss: 0.42734160122840353
Epoch 1, Accuracy: 0.9466, Train Loss: 0.2186828709722031


KeyboardInterrupt: 